# BirdCLEF+ 2026 — Perch v2 Embedding Extraction

Precomputes Perch v2 outputs for everything we'll train on. Two outputs per audio window:

| Output | Shape | Use |
|---|---|---|
| `embedding` | `(N, 1536)` | Phase 3 MLP head — single vector per clip |
| `spatial_embedding` (mean-pooled over freq) | `(N, 16, 1536)` | **Phase 4 SED head** — 16 timesteps per clip |

The SED head's attention mechanism operates on the 16 timesteps, so we need the per-timestep version.

**Output files** (in `embeddings/`):

| File | Shape | Size |
|---|---|---|
| `clip_embeddings.npy` | (35549, 1536) | ~210 MB |
| `clip_index.csv` | — | small |
| `ss_window_embeddings.npy` | (1478, 1536) | ~9 MB |
| `ss_window_index.csv` | — | small |
| `spatial_emb_clips.npy` | (35549, 16, 1536) | ~3.5 GB |
| `spatial_emb_ss.npy` | (1478, 16, 1536) | ~140 MB |

**Total runtime**: ~30-50 min (Perch CPU inference at ~32 wins/s). Resumable — each cell checkpoints periodically.


## 1. Setup


In [ ]:
import os, time
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import onnxruntime as ort
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data" / "birdclef-2026"
TRAIN_AUDIO  = DATA_DIR / "train_audio"
SS_AUDIO_DIR = DATA_DIR / "train_soundscapes"
PERCH_PATH   = PROJECT_ROOT / "data" / "perch" / "perch_v2_no_dft.onnx"
EMBED_DIR    = PROJECT_ROOT / "embeddings"
EMBED_DIR.mkdir(exist_ok=True)

assert PERCH_PATH.exists(), f"Missing {PERCH_PATH} — run kaggle datasets download tuckerarrants/perch-v2-no-dft-onnx first"

SR        = 32000
N_SAMPLES = SR * 5   # 5-second window = 160,000 samples
EMBED_DIM = 1536

print(f"Perch ONNX: {PERCH_PATH} ({PERCH_PATH.stat().st_size/1e6:.0f} MB)")


## 2. Load Perch + identify output indices

Perch v2 returns 4 outputs. We use two:
- **`embedding`** — global 1536-d vector per clip (averaged across the entire 5s)
- **`spatial_embedding`** — `(16 time, 4 freq, 1536)` — per-timestep, per-freq features. We mean-pool over freq to get `(16, 1536)`.


In [ ]:
sess = ort.InferenceSession(str(PERCH_PATH), providers=["CPUExecutionProvider"])
INPUT_NAME = sess.get_inputs()[0].name

EMB_IDX     = next(i for i, o in enumerate(sess.get_outputs()) if o.name == "embedding")
SPATIAL_IDX = next(i for i, o in enumerate(sess.get_outputs()) if o.name == "spatial_embedding")
print(f"Perch input: {INPUT_NAME}  shape={sess.get_inputs()[0].shape}")
print(f"  embedding output idx={EMB_IDX}")
print(f"  spatial_embedding output idx={SPATIAL_IDX}")

# Smoke test
_dummy = np.random.randn(1, N_SAMPLES).astype(np.float32)
_out = sess.run(None, {INPUT_NAME: _dummy})
print(f"  embedding shape: {_out[EMB_IDX].shape}")
print(f"  spatial_embedding shape: {_out[SPATIAL_IDX].shape}")


## 3. Audio helpers


In [ ]:
def load_audio(path):
    wav, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    return wav.astype(np.float32)


def take_center_5s(wav):
    if len(wav) < N_SAMPLES:
        pad = N_SAMPLES - len(wav)
        wav = np.pad(wav, (pad // 2, pad - pad // 2))
    elif len(wav) > N_SAMPLES:
        s = (len(wav) - N_SAMPLES) // 2
        wav = wav[s:s + N_SAMPLES]
    return wav.astype(np.float32)


def parse_time_to_sec(s):
    h, m, ss = s.split(":")
    return int(h) * 3600 + int(m) * 60 + int(ss)


def slice_window(wav, start_sec, end_sec):
    start_samp, end_samp = start_sec * SR, end_sec * SR
    if len(wav) < end_samp:
        wav = np.pad(wav, (0, end_samp - len(wav)))
    chunk = wav[start_samp:end_samp]
    if len(chunk) < N_SAMPLES:
        chunk = np.pad(chunk, (0, N_SAMPLES - len(chunk)))
    elif len(chunk) > N_SAMPLES:
        chunk = chunk[:N_SAMPLES]
    return chunk.astype(np.float32)


## 4. Extract clip embeddings (both outputs in one pass)

For each clip in `train.csv`: take the center 5 seconds, run Perch, save both
the global `embedding` and the mean-pooled `spatial_embedding`.

Resumable — saves every 2,000 clips.


In [ ]:
df = pd.read_csv(DATA_DIR / "train.csv")
df["primary_label"] = df["primary_label"].astype(str)
N_CLIPS = len(df)
print(f"Total clips: {N_CLIPS:,}")

clip_emb_path     = EMBED_DIR / "clip_embeddings.npy"
clip_spatial_path = EMBED_DIR / "spatial_emb_clips.npy"
clip_index_path   = EMBED_DIR / "clip_index.csv"
clip_progress     = EMBED_DIR / "clip_progress.txt"

if clip_emb_path.exists() and clip_spatial_path.exists() and clip_progress.exists():
    clip_emb     = np.load(clip_emb_path)
    clip_spatial = np.load(clip_spatial_path)
    start_idx    = int(clip_progress.read_text().strip())
    print(f"Resuming from clip {start_idx}/{N_CLIPS}")
else:
    clip_emb     = np.zeros((N_CLIPS, EMBED_DIM), dtype=np.float32)
    clip_spatial = np.zeros((N_CLIPS, 16, EMBED_DIM), dtype=np.float32)
    start_idx    = 0

df[["filename", "primary_label", "secondary_labels"]].to_csv(clip_index_path, index=False)

BATCH = 8
SAVE_EVERY = 2000

t0 = time.time()
pbar = tqdm(total=N_CLIPS, initial=start_idx, desc="clips")
i = start_idx
while i < N_CLIPS:
    batch_idx = list(range(i, min(i + BATCH, N_CLIPS)))
    batch_wavs = []
    for j in batch_idx:
        path = TRAIN_AUDIO / df.iloc[j]["filename"]
        try:
            wav = take_center_5s(load_audio(path))
        except Exception as e:
            print(f"[skip] {path}: {e}")
            wav = np.zeros(N_SAMPLES, dtype=np.float32)
        batch_wavs.append(wav)
    outs = sess.run(None, {INPUT_NAME: np.stack(batch_wavs)})
    clip_emb[i:i+len(batch_idx)]     = outs[EMB_IDX]                # (B, 1536)
    clip_spatial[i:i+len(batch_idx)] = outs[SPATIAL_IDX].mean(axis=2)  # (B, 16, 4, 1536) -> mean freq -> (B, 16, 1536)
    i += len(batch_idx)
    pbar.update(len(batch_idx))
    if i % SAVE_EVERY < BATCH:
        np.save(clip_emb_path, clip_emb)
        np.save(clip_spatial_path, clip_spatial)
        clip_progress.write_text(str(i))
pbar.close()

np.save(clip_emb_path, clip_emb)
np.save(clip_spatial_path, clip_spatial)
clip_progress.write_text(str(i))
print(f"\nClips done in {(time.time()-t0)/60:.1f} min")
print(f"  clip_embeddings.npy:   {clip_emb.shape}  {clip_emb.nbytes/1e6:.0f} MB")
print(f"  spatial_emb_clips.npy: {clip_spatial.shape}  {clip_spatial.nbytes/1e9:.1f} GB")


## 5. Extract labeled-soundscape window embeddings

For each labeled window in `train_soundscapes_labels.csv`: load the soundscape file,
slice the exact 5s window, run Perch, save both outputs.

Optimization: each soundscape file appears in many label rows. We cache the loaded
waveform across rows to avoid re-reading the file.


In [ ]:
ss_df = pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv")
ss_df["start_sec"] = ss_df["start"].apply(parse_time_to_sec)
ss_df["end_sec"]   = ss_df["end"].apply(parse_time_to_sec)
N_SS = len(ss_df)
print(f"Labeled SS windows: {N_SS:,}  ({ss_df['filename'].nunique()} unique files)")

ss_emb_path     = EMBED_DIR / "ss_window_embeddings.npy"
ss_spatial_path = EMBED_DIR / "spatial_emb_ss.npy"
ss_index_path   = EMBED_DIR / "ss_window_index.csv"
ss_progress     = EMBED_DIR / "ss_progress.txt"

if ss_emb_path.exists() and ss_spatial_path.exists() and ss_progress.exists():
    ss_emb     = np.load(ss_emb_path)
    ss_spatial = np.load(ss_spatial_path)
    start_idx  = int(ss_progress.read_text().strip())
    print(f"Resuming from {start_idx}/{N_SS}")
else:
    ss_emb     = np.zeros((N_SS, EMBED_DIM), dtype=np.float32)
    ss_spatial = np.zeros((N_SS, 16, EMBED_DIM), dtype=np.float32)
    start_idx  = 0

ss_df[["filename", "start_sec", "end_sec", "primary_label"]].to_csv(ss_index_path, index=False)

t0 = time.time()
pbar = tqdm(total=N_SS, initial=start_idx, desc="ss windows")
i = start_idx
current_file, current_wav = None, None
while i < N_SS:
    batch_idx, batch_wavs = [], []
    while len(batch_idx) < BATCH and i + len(batch_idx) < N_SS:
        row = ss_df.iloc[i + len(batch_idx)]
        if row["filename"] != current_file:
            try:
                current_wav  = load_audio(SS_AUDIO_DIR / row["filename"])
                current_file = row["filename"]
            except Exception as e:
                print(f"[skip] {row['filename']}: {e}")
                current_wav = np.zeros(60 * SR, dtype=np.float32)
        chunk = slice_window(current_wav, row["start_sec"], row["end_sec"])
        batch_idx.append(i + len(batch_idx))
        batch_wavs.append(chunk)

    outs = sess.run(None, {INPUT_NAME: np.stack(batch_wavs)})
    ss_emb[batch_idx[0]:batch_idx[0]+len(batch_wavs)]     = outs[EMB_IDX]
    ss_spatial[batch_idx[0]:batch_idx[0]+len(batch_wavs)] = outs[SPATIAL_IDX].mean(axis=2)
    i = batch_idx[-1] + 1
    pbar.update(len(batch_wavs))
    if i % 200 < BATCH:
        np.save(ss_emb_path, ss_emb)
        np.save(ss_spatial_path, ss_spatial)
        ss_progress.write_text(str(i))
pbar.close()

np.save(ss_emb_path, ss_emb)
np.save(ss_spatial_path, ss_spatial)
ss_progress.write_text(str(i))
print(f"\nSS windows done in {(time.time()-t0)/60:.1f} min")
print(f"  ss_window_embeddings.npy: {ss_emb.shape}")
print(f"  spatial_emb_ss.npy:       {ss_spatial.shape}")


## 6. Sanity check

Same-species clips should have higher cosine similarity than diff-species clips
in Perch's embedding space.


In [ ]:
def cos_sim(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)


clip_idx_df = pd.read_csv(clip_index_path)
clip_idx_df["primary_label"] = clip_idx_df["primary_label"].astype(str)
nonzero = (np.abs(clip_emb).sum(axis=1) > 0).sum()
print(f"Non-zero global embeddings: {nonzero}/{len(clip_emb)}")

target = clip_idx_df["primary_label"].value_counts().head(1).index[0]
same_idx = clip_idx_df[clip_idx_df["primary_label"] == target].index[:10].tolist()
diff_idx = clip_idx_df[clip_idx_df["primary_label"] != target].sample(10, random_state=0).index.tolist()

same = [cos_sim(clip_emb[same_idx[0]], clip_emb[j]) for j in same_idx[1:]]
diff = [cos_sim(clip_emb[same_idx[0]], clip_emb[j]) for j in diff_idx]
print(f"\nSpecies {target}:")
print(f"  Same-species cos sim: mean={np.mean(same):.3f}  range=[{min(same):.3f}, {max(same):.3f}]")
print(f"  Diff-species cos sim: mean={np.mean(diff):.3f}  range=[{min(diff):.3f}, {max(diff):.3f}]")
print(f"\nSame > Diff? {np.mean(same) > np.mean(diff)}")


## What's next

When this completes, the embeddings power both training paths:

- **`01_train.ipynb`** — Phase 4 SED + Phase 6 K-fold (uses `spatial_emb_*.npy`)
- The older Phase 3 MLP-head approach uses the global `clip_embeddings.npy` / `ss_window_embeddings.npy`

Phase 4 + 6 combined gave LB **0.842** (current best).
